In [1]:
import pandas as pd  
import numpy as np  
import os  
import glob  
import warnings

warnings.filterwarnings('ignore', category=UserWarning, module='openpyxl')

In [2]:
pc_folder_path = 'L:/2026 Pilar Plant Files/Position Control'

# --- adjust date ---    
file_date = '2026-09-12'  
# -------------------

# get all matching files    
pc_files = glob.glob(os.path.join(pc_folder_path, f'*{file_date}*.xlsx'))

print(f"Found {len(pc_files)} files matching date {file_date}")

site_pc_list = []

for file in pc_files:    
    filename = os.path.basename(file)    
      
    try:  
        # extract site (first 5 characters of filename)    
        site = filename[:5]    
          
        # read second tab, skip first 12 rows    
        file_data = pd.read_excel(file, sheet_name='PositionDetail', skiprows=12)    
          
        # add site and date columns    
        file_data['BU'] = site    
        file_data['Date'] = file_date  

        f_file_data = file_data[['Rpt Dept', 'Rpt Dept Desc', 'Job Code', 'Jobcode Title', 'Job Function / Family', 'Posn Number',   
                                'Posn Status', 'Filled/ Open', 'Incumbent Name', 'Emplid', 'Incumbent Status',  
                                'Reg/ Temp', 'Posn Type', 'Filled Hrs', 'Filled FTE', 'Open Hrs', 'Open FTE', 'BU', 'Date']]

        fc_file_data = f_file_data[f_file_data['Rpt Dept'].notna()]  
          
        site_pc_list.append(fc_file_data)  
        print(f"Processed: {filename}")  
      
    except PermissionError:  
        print(f"SKIPPED (permission denied): {filename}")  
        continue

# combine all into one dataframe    
if site_pc_list:    
    position_control = pd.concat(site_pc_list, ignore_index=True)    
    print(f"\nTotal rows: {len(position_control)}")    
else:    
    print(f"No files found for date: {file_date}")  

Found 4 files matching date 2026-09-12


Processed: GVCCC POSNEXEC GVCCC_EXEC 2026-09-12.xlsx
Processed: LENOX POSNEXEC LENOX_EXEC 2026-09-12.xlsx
Processed: MEETH POSNEXEC MEETH_EXEC 2026-09-12.xlsx
SKIPPED (permission denied): ~$LENOX POSNEXEC LENOX_EXEC 2026-09-12.xlsx

Total rows: 6271


In [3]:
position_control['filled_active'] = np.where(position_control['Incumbent Status'] == 'A', position_control['Filled FTE'], 0) 
position_control['loa'] = np.where(position_control['Incumbent Status'].isin(['L', 'P']), position_control['Filled FTE'], 0)
position_control['open'] = np.where(position_control['Filled/ Open'] == 'Open', position_control['Open FTE'], 0)
position_control['total_ftes'] = position_control['filled_active'] + position_control['loa'] + position_control['open']

pc_values = position_control[position_control['total_ftes'] != 0]
pc_full = pc_values.groupby(['BU', 'Date', 'Rpt Dept', 'Job Code'])[['filled_active', 'loa', 'open', 'total_ftes']].sum().reset_index()

pc_lookup = pc_values[['Rpt Dept', 'Rpt Dept Desc', 'Job Code', 'Jobcode Title']].drop_duplicates().reset_index()

display(pc_full[['filled_active', 'loa', 'open', 'total_ftes']].sum())
display(pc_full[['Rpt Dept', 'Job Code']].drop_duplicates().count())

filled_active    3946.82
loa               195.18
open              520.16
total_ftes       4662.16
dtype: float64

Rpt Dept    1018
Job Code    1018
dtype: int64

In [4]:
req_path_name = 'L:/2026 Pilar Plant Files/Requisition Reports/'
req_file_name = 'NYC Req Reports 9.10.2026.xlsx'
pending_req_tab = 'Pending Approval'
approved_req_tab = 'Open'

file = req_path_name + req_file_name

# read two tabs into separate dataframes  
pending_reqs = pd.read_excel(file, sheet_name=pending_req_tab) 
approved_reqs = pd.read_excel(file, sheet_name=approved_req_tab)   

approved_reqs_site_spec = approved_reqs[approved_reqs['BUSINESS UNIT'].isin(['LENOX', 'MEETH', 'GVCCC'])]
approved_reqs_full = approved_reqs_site_spec.groupby(['BUSINESS UNIT', 'DEPARTMENT NUMBER', 'JOB CODE'])['FTE'].sum().reset_index()

pending_reqs_site_spec = pending_reqs[pending_reqs['BUSINESS UNIT'].isin(['LENOX', 'MEETH', 'GVCCC'])]
pending_reqs_full = pending_reqs_site_spec.groupby(['BUSINESS UNIT', 'DEPARTMENT NUMBER', 'JOB CODE', 'NEW REPLACE'])['FTE'].sum().reset_index()

pending_reqs_piv = pending_reqs_full.pivot_table(  
    index=['BUSINESS UNIT', 'DEPARTMENT NUMBER', 'JOB CODE'],
    columns='NEW REPLACE',
    values='FTE',
).reset_index()

pending_reqs_piv.columns.name = None 

In [5]:
# Check for duplicate merge keys on each side  
pc_dupes = pc_full[pc_full.duplicated(subset=['Rpt Dept', 'Job Code'], keep=False)]  
req_dupes = approved_reqs_full[approved_reqs_full.duplicated(subset=['DEPARTMENT NUMBER', 'JOB CODE'], keep=False)]

print(f"pc_full duplicate dept/jc combos: {len(pc_dupes)}")  
print(f"approved_reqs_full duplicate dept/jc combos: {len(req_dupes)}")

if not pc_dupes.empty:  
    display(pc_dupes.sort_values(['Rpt Dept', 'Job Code']).head(10))  
if not req_dupes.empty:  
    display(req_dupes.sort_values(['DEPARTMENT NUMBER', 'JOB CODE']).head(10))

pc_full duplicate dept/jc combos: 0
approved_reqs_full duplicate dept/jc combos: 0


In [6]:
pc_app_reqs = pd.merge(pc_full, approved_reqs_full, how='outer', left_on=['Rpt Dept', 'Job Code'], right_on=['DEPARTMENT NUMBER', 'JOB CODE'])

pc_app_reqs['dept_id'] = np.where(pc_app_reqs['Rpt Dept'].notna(), pc_app_reqs['Rpt Dept'], pc_app_reqs['DEPARTMENT NUMBER'])
pc_app_reqs['bu'] = np.where(pc_app_reqs['BU'].notna(), pc_app_reqs['BU'], pc_app_reqs['BUSINESS UNIT'])  
pc_app_reqs['job_code'] = np.where(pc_app_reqs['Job Code'].notna(), pc_app_reqs['Job Code'], pc_app_reqs['JOB CODE'])  
pc_app_reqs['approved_ftes'] = pc_app_reqs['FTE']

pc_app_reqs_merged = pc_app_reqs[['bu', 'dept_id', 'job_code', 'filled_active', 'loa', 'open', 'total_ftes','approved_ftes']]

display(pc_app_reqs_merged[['approved_ftes']].sum())


approved_ftes    128.704762
dtype: float64

In [7]:
pc_app_pend_reqs = pd.merge(pc_app_reqs_merged, pending_reqs_piv, how='outer', left_on=['dept_id', 'job_code'], right_on=['DEPARTMENT NUMBER', 'JOB CODE'])

pc_app_pend_reqs['dept_id'] = np.where(pc_app_pend_reqs['dept_id'].notna(), pc_app_pend_reqs['dept_id'], pc_app_pend_reqs['DEPARTMENT NUMBER'])
pc_app_pend_reqs['bu'] = np.where(pc_app_pend_reqs['bu'].notna(), pc_app_pend_reqs['bu'], pc_app_pend_reqs['BUSINESS UNIT'])  
pc_app_pend_reqs['job_code'] = np.where(pc_app_pend_reqs['job_code'].notna(), pc_app_pend_reqs['job_code'], pc_app_pend_reqs['JOB CODE'])  
pc_app_pend_reqs['new_ftes'] = pc_app_pend_reqs['NEW']
pc_app_pend_reqs['replacement_ftes'] = pc_app_pend_reqs['REPLACE']
# might need position change column tbd

pc_reqs_full = pc_app_pend_reqs.fillna(0)

pc_reqs_full['total_req_ftes'] = pc_reqs_full['approved_ftes'] + pc_reqs_full['new_ftes'] + pc_reqs_full['replacement_ftes']

pc_reqs_merged = pc_reqs_full[['bu', 'dept_id', 'job_code', 'filled_active', 'loa', 'open', 'total_ftes', 'approved_ftes', 'new_ftes', 'replacement_ftes', 'total_req_ftes']]

In [9]:
# pull xwalk reference file  
xwalk = pd.read_excel("C:/Users/kbixby/OneDrive - Northwell Health/Scripts/fte/dept_jc_lookup_table.xlsx")

# check for duplicates in crosswalk    
dupes = xwalk[xwalk.duplicated(subset=['dept', 'jc'], keep=False)]  
if not dupes.empty:    
    print(f"WARNING: {len(dupes)} duplicate dept/jc rows found in crosswalk:")    
    display(dupes.sort_values(['dept', 'jc']))

# set conditions for replacement (dept_id, job_code, new_dept_id)  
dept_jc_conditions = [  
    (15600360, 116649, 15601055),  
    (15600360, '~116649', 15600180),  
    (29600360, 116649, 29601055),  
    (29600360, '~116649', 29600180),
]

# replace dept id and desc where dept id and jc match
for dept_id, jc, new_dept_id in dept_jc_conditions:  
    if isinstance(jc, str) and jc.startswith('~'):  
        mask = (pc_reqs_merged['dept_id'] == dept_id) & (pc_reqs_merged['job_code'] != int(jc[1:]))  
    else:  
        mask = (pc_reqs_merged['dept_id'] == dept_id) & (pc_reqs_merged['job_code'] == jc)  
    pc_reqs_merged.loc[mask, 'dept_id'] = new_dept_id  

# merge department leadership from xwalk  
dept_lookup = xwalk[['vp', 'director', 'dept', 'dept_desc']].drop_duplicates(subset=['dept'], keep='first')  
pc_reqs_dept = pd.merge(dept_lookup, pc_reqs_merged, how='right', left_on='dept', right_on='dept_id')  

pc_reqs_dept['dept_id'] = pc_reqs_dept['dept_id'].fillna(pc_reqs_dept['dept'])    
pc_reqs_dept = pc_reqs_dept.drop(columns=['dept']) 

# remove corporate retained and employee health services  
pc_depts_f = pc_reqs_dept[~pc_reqs_dept['dept_desc'].str.contains(  
    'Corporate Retained|Corp Retained|Employee Health Svcs', case=False, na=False)].copy()

# set conditions for replacement (dept_id, new_dept_desc)  
dept_conditions = [  
    (74002564, 'POS - Perianesthesia'),  
    (74002545, 'POS - Operating Room'),  
    (15600100, 'Hospital Material'),  
    (15600160, 'Periop Material')  
]

for dept_id, new_dept_desc in dept_conditions:    
    mask = (pc_depts_f['dept_id'] == dept_id)    
    pc_depts_f.loc[mask, 'dept_desc'] = new_dept_desc 

pc_grouped = pc_depts_f.groupby(['bu', 'vp', 'director', 'dept_id', 'dept_desc', 'job_code'])[    
    ['filled_active', 'loa', 'open', 'total_ftes', 'approved_ftes', 'new_ftes', 'replacement_ftes', 'total_req_ftes']    
].sum().reset_index()  


# pull job description from xwalk  
jc_lookup = xwalk[['dept', 'jc', 'jc_desc']].drop_duplicates(subset=['dept', 'jc'], keep='first')  
pc_req_ftes = pd.merge(pc_grouped, jc_lookup, how='left', left_on=['dept_id', 'job_code'], right_on=['dept', 'jc'])  
pc_req_ftes = pc_req_ftes.drop(columns=['dept', 'jc'])

# pull jobcode title from pc_lookup  
pc_lookup_dedup = pc_lookup.drop_duplicates(subset=['Rpt Dept', 'Job Code'], keep='first')  
pc_req_ftes_jc = pd.merge(pc_req_ftes, pc_lookup_dedup, how='left', left_on=['dept_id', 'job_code'], right_on=['Rpt Dept', 'Job Code'])  
pc_req_ftes_jc = pc_req_ftes_jc.drop(columns=['Rpt Dept', 'Job Code']) 

pc_req_ftes_jc['jc_desc'] = np.where(  
    pc_req_ftes_jc['jc_desc'].notna() & (pc_req_ftes_jc['jc_desc'].str.strip() != ''),   
    pc_req_ftes_jc['jc_desc'],   
    pc_req_ftes_jc['Jobcode Title']  
)  
pc_req_ftes_jc['position_change'] = 0
pc_ftes = pc_req_ftes_jc[['bu', 'vp', 'director', 'dept_id', 'dept_desc', 'job_code', 'jc_desc', 'filled_active', 'loa', 'open', 'total_ftes', 
                        'new_ftes', 'replacement_ftes', 'position_change', 'approved_ftes', 'total_req_ftes']]

# display summary  
display(pc_ftes[['filled_active', 'loa', 'open', 'total_ftes', 'approved_ftes',   
                     'new_ftes', 'replacement_ftes', 'total_req_ftes']].sum())

# export  
pc_ftes.to_excel("positioncontrol.xlsx", index=False)  

# compare final output to xwalk and append new dept/jc combos  
new_xwalk_rows = pc_ftes[['vp', 'director', 'bu', 'dept_id', 'dept_desc', 'job_code', 'jc_desc']].drop_duplicates()  
new_xwalk_rows = new_xwalk_rows.rename(columns={'dept_id': 'dept', 'job_code': 'jc'})

# filter to only rows not already in xwalk  
existing = xwalk[['dept', 'jc']].drop_duplicates()  
new_xwalk_rows = pd.merge(new_xwalk_rows, existing, on=['dept', 'jc'], how='left', indicator=True)  
new_xwalk_rows = new_xwalk_rows[new_xwalk_rows['_merge'] == 'left_only'].drop(columns='_merge')

if not new_xwalk_rows.empty:  
    # reorder to match xwalk columns  
    xwalk_cols = [col for col in xwalk.columns if col in new_xwalk_rows.columns]  
    new_xwalk_rows = new_xwalk_rows[xwalk_cols]  
      
    xwalk = pd.concat([xwalk, new_xwalk_rows], ignore_index=True).drop_duplicates()  
    xwalk.to_excel("C:/Users/kbixby/OneDrive - Northwell Health/Scripts/fte/dept_jc_lookup_table.xlsx", index=False)  
    print(f"Added {len(new_xwalk_rows)} new dept/jc combos to crosswalk from position control.")  
    display(new_xwalk_rows)  
else:  
    print("No new dept/jc combos to add to crosswalk.")

# export    
pc_ftes.to_excel("positioncontrol.xlsx", index=False)

filled_active       3946.320000
loa                  195.180000
open                 520.160000
total_ftes          4661.660000
approved_ftes        128.704762
new_ftes               5.500000
replacement_ftes      38.926667
total_req_ftes       173.131428
dtype: float64

No new dept/jc combos to add to crosswalk.
